<a href="https://www.kaggle.com/code/tirendazacademy/transformers-with-pytorch?scriptVersionId=143972938" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# <b><div style='padding:15px;background-color:#850E35;color:white;border-radius:2px;font-size:110%;text-align: center'>Transformers with PyTorch</div></b>
![](https://pytorch.org/assets/images/pytorch-2.0-feature-img.png)

This notebook walks you through how to work with Transformers using PyTorch.

# <b><div style='padding:15px;background-color:#850E35;color:white;border-radius:2px;font-size:110%;text-align: center'>1. Loading Dataset</div></b>

Let's install the "datasets" library using pip. This library is commonly used for accessing and working with various datasets.

In [ ]:
# Installing datasets:
# !pip install -q datasets

Let's import the load_dataset function from the "datasets" library and use it to load the "rotten_tomatoes" dataset. 

In [1]:
from datasets import load_dataset

# Loading the rotten_tomatoes dataset:
dataset = load_dataset("rotten_tomatoes")

README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

# <b><div style='padding:15px;background-color:#850E35;color:white;border-radius:2px;font-size:110%;text-align: center'>2. Model Loading</div></b>

Let's import the AutoModelForSequenceClassification class from the "transformers" library and initialize a pre-trained sequence classification model called "distilbert-base-uncased."

In [2]:
from transformers import AutoModelForSequenceClassification

# Loading DistilBERT:
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

2026-01-01 16:11:54.535614: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767283914.984480      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767283915.112184      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767283916.245820      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767283916.245864      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767283916.245867      55 computation_placer.cc:177] computation placer alr

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# <b><div style='padding:15px;background-color:#850E35;color:white;border-radius:2px;font-size:110%;text-align: center'>3. Tokenization</div></b>

Let's first imports the AutoTokenizer class and initialize a tokenizer that corresponds to the previously initialized model.

In [3]:
from transformers import AutoTokenizer

# Loading tokenizer for DistilBERT:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Let's create a function named tokenize_dataset. It takes a dataset as input and tokenizes the "text" column of the dataset using the previously created tokenizer. And then let's use to apply this tokenization function to the entire dataset in batches.

In [4]:
# Creating a function to tokenize the dataset:
def tokenize_dataset(dataset):
    return tokenizer(dataset["text"])

# Applying tokenizer to all dataset:
dataset = dataset.map(tokenize_dataset, batched=True)

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

# <b><div style='padding:15px;background-color:#850E35;color:white;border-radius:2px;font-size:110%;text-align: center'>4. Padding</div></b>

Let me import the DataCollatorWithPadding class and initialize a data collator that will be used to batch and pad the tokenized data.

In [5]:
from transformers import DataCollatorWithPadding

# Creating a data collator that will dynamically pad the inputs received:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# <b><div style='padding:15px;background-color:#850E35;color:white;border-radius:2px;font-size:110%;text-align: center'>5. Setting Training Arguments</div></b>

Let's set up the training arguments, specifying details such as the output directory for model checkpoints, learning rate, batch sizes, number of training epochs, and reporting options.

In [6]:
from transformers import TrainingArguments

# Creating training arguments that contain the model hyperparameters:
training_args = TrainingArguments(
    output_dir="my_bert_model",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    report_to="none",
)

# <b><div style='padding:15px;background-color:#850E35;color:white;border-radius:2px;font-size:110%;text-align: center'>6. Model Training</div></b>

Next, let's first import the Trainer class and initialize a trainer object. This trainer is used to train the model using the specified training arguments, datasets, tokenizer, and data collator.

In [7]:
from transformers import Trainer

# Gathering all these classes in Trainer:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,   
)

/tmp/ipykernel_55/1531935842.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Let's call train() to start training.

In [8]:
# Calling train() to start training:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
500,0.425000
1000,0.254600


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=1068, training_loss=0.33266260204243747, metrics={'train_runtime': 106.6964, 'train_samples_per_second': 159.893, 'train_steps_per_second': 10.01, 'total_flos': 215637261882480.0, 'train_loss': 0.33266260204243747, 'epoch': 2.0})

# <b><div style='padding:15px;background-color:#850E35;color:white;border-radius:2px;font-size:110%;text-align: center'>7. Prediction</div></b>

To predict, let's create a text.

In [9]:
# Getting a text for prediction:
text = "I love NLP. It's fun to analyze the NLP tasks with Hugging Face"

Let's tokenize text and store in the inputs variable.

In [10]:
# Preprocessing the text:
inputs = tokenizer(text, return_tensors="pt")
inputs

{'input_ids': tensor([[  101,  1045,  2293, 17953,  2361,  1012,  2009,  1005,  1055,  4569,
          2000, 17908,  1996, 17953,  2361,  8518,  2007, 17662,  2227,   102]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

To calculate predictions, first, let's import the PyTorch library, and load the fine-tuned model from a specified path. This fine-tuned model is used for inference.

In [11]:
import torch

# Loading the model from the file:
model_path = "/kaggle/working/my_bert_model/checkpoint-1000"
model = AutoModelForSequenceClassification.from_pretrained(
    model_path, num_labels=2)

# Calculating predictions:
with torch.no_grad():
    logits = model(**inputs).logits

Let's take a look at the class with the highest logit score as the predicted class. The result is stored in the predicted_class_id variable. This is essentially performing a classification task where the model predicts a class label based on the input text.

In [12]:
# Looking the prediction:
predicted_class_id = logits.argmax().item()
predicted_class_id

1

Thanks for reading. If you enjoyed this notebook, don't forget to upvote 👍

Let's connect [YouTube](http://youtube.com/tirendazacademy) | [Medium](http://tirendazacademy.medium.com) | [Twitter](http://twitter.com/tirendazacademy) | [Instagram](https://www.instagram.com/tirendazacademy) | [GitHub](http://github.com/tirendazacademy) | [Linkedin](https://www.linkedin.com/in/tirendaz-academy) | [Kaggle](https://www.kaggle.com/tirendazacademy) 😎